## semantic chunking

In [1]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

d:\RAG System\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
with open('sample.txt', 'r') as file:
    text= file.read()
print(text)    

Machine learning is a field of artificial intelligence that focuses on building systems that can learn from data. It includes techniques such as supervised learning, unsupervised learning, and reinforcement learning. Applications range from image recognition to natural language processing.

Climate change refers to long-term shifts in temperatures and weather patterns, mainly caused by human activities like burning fossil fuels. It leads to rising sea levels, extreme weather events, and loss of biodiversity. Governments and organizations are working on mitigation strategies and renewable energy adoption.

The history of space exploration began with the launch of Sputnik in 1957. Since then, humans have sent satellites, probes, and astronauts to explore space. Key milestones include landing on the Moon, establishing the International Space Station, and sending rovers to Mars.

Nutrition is the study of how food affects the health and growth of the human body. A balanced diet includes ca

In [13]:
## Initialize the model
model=SentenceTransformer('all-MiniLM-L6-v2')


## Step 1 : Split into sentences
sentences=[s.strip() for s in text.split("\n") if s.strip()]

### sstep 2: Embed each setence
embeddings=model.encode(sentences)

# Step 3: Initialize parameters
threshold = 0.7  # control chunk tightness
chunks = []
current_chunk=[sentences[0]]

## Step 4: Semantic grouping based on threshold

for i in range(1, len(sentences)):
    sim = cosine_similarity(
        [embeddings[i - 1]],
        [embeddings[i]]
    )[0][0]

    if sim>=threshold:
        current_chunk.append(sentences[i])
    else:
        chunks.append(" ".join(current_chunk))
        current_chunk=[sentences[i]]

# Append the last chunk
chunks.append(" ".join(current_chunk))

# Output the chunks
print("\n📌 Semantic Chunks:")
for idx, chunk in enumerate(chunks):
    print(f"\nChunk {idx+1}:\n{chunk}")






Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1343.17it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



📌 Semantic Chunks:

Chunk 1:
Machine learning is a field of artificial intelligence that focuses on building systems that can learn from data. It includes techniques such as supervised learning, unsupervised learning, and reinforcement learning. Applications range from image recognition to natural language processing.

Chunk 2:
Climate change refers to long-term shifts in temperatures and weather patterns, mainly caused by human activities like burning fossil fuels. It leads to rising sea levels, extreme weather events, and loss of biodiversity. Governments and organizations are working on mitigation strategies and renewable energy adoption.

Chunk 3:
The history of space exploration began with the launch of Sputnik in 1957. Since then, humans have sent satellites, probes, and astronauts to explore space. Key milestones include landing on the Moon, establishing the International Space Station, and sending rovers to Mars.

Chunk 4:
Nutrition is the study of how food affects the health 

### semnatic chunk with langchain

In [17]:
from langchain_community.document_loaders import TextLoader
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings
loader=TextLoader('sample.txt')
documents=loader.load()
embeddings=HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')
chunker=SemanticChunker(embeddings=embeddings)
chunks=chunker.split_documents(documents)
print("\n📌 Semantic Chunks:")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: {chunk}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2616.71it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



📌 Semantic Chunks:
Chunk 1: page_content='Machine learning is a field of artificial intelligence that focuses on building systems that can learn from data. It includes techniques such as supervised learning, unsupervised learning, and reinforcement learning. Applications range from image recognition to natural language processing. Climate change refers to long-term shifts in temperatures and weather patterns, mainly caused by human activities like burning fossil fuels. It leads to rising sea levels, extreme weather events, and loss of biodiversity.' metadata={'source': 'sample.txt'}
Chunk 2: page_content='Governments and organizations are working on mitigation strategies and renewable energy adoption. The history of space exploration began with the launch of Sputnik in 1957. Since then, humans have sent satellites, probes, and astronauts to explore space. Key milestones include landing on the Moon, establishing the International Space Station, and sending rovers to Mars. Nutrition is 